# Order book program


In [ ]:
import numpy as np
import pandas as pd
import queue as Q
import os 
import sys
import pickle
import matplotlib.pyplot as plt


#
# When debugging, I print out stuff to see what's going on
# I would not turn debugging on for the entire set of requests,
# but I would test on a smaller set of requests i.e. stop processing
# after a small number.
#
DEBUG=False
if DEBUG:
    forderbook=open("orderbooks.txt","w")
    forderbook.close
    number_of_requests_to_process=1000

ftrades=open("TRADES.csv","w")
ftrades.close()
    
    
trades_processed=0 # track number of trades processed

trade_desc=""


#
# Dictionary to look up a bid from a bidid
#
bid_dictionary={}
ask_dictionary={}

#
# Dictionary of customers
#
customer_dictionary={}

#
# The bid and ask priority queues.
#
bid_queue=Q.PriorityQueue()
ask_queue=Q.PriorityQueue()

#
# Functions I use to format my own time of day object
#
def mystr2(n):
    if n<10:
        return("0"+str(n))
    return(str(n))
def mystr3(n):
    if n==0:
        return("000000")
    if n<10:
        return("00000"+str(n))
    if n<100:
        return("0000"+str(n))
    if n<1000:
        return("000"+str(n))
    if n<10000:
        return("00"+str(n))
    if n<100000:
        return("0"+str(n))
    return(str(n))    
#
# time of day object
#
class timeofday:
    def __init__(self,hr,minute,sec,ms):
        self.hr=hr
        self.min=minute
        self.sec=sec
        self.ms=ms
    def __str__(self):
        tstr=mystr2(self.hr)+":"
        tstr+=mystr2(self.min)+":"
        tstr+=mystr2(self.sec)+":"
        tstr+=mystr3(self.ms)
        return(tstr[0:-3]) 
#
# Problem 1. Write a function __lt__ that 
# returns true of self timeofday is at or before 
# other timeofday
#
    def __lt__(self,other):
        if self.hr == other.hr:
            if self.min == other.min:
                if self.sec == other.sec:
                    return self.ms < other.ms
                else:
                    return self.sec < other.sec
            else:
                return self.min < other.min
        else:
            return self.hr < other.hr
        ...

#
# request object
#
class request:
    # constructor takes string read from requests file 
    def __init__(self,line):
        L=line.split(",")
        hr=int(L[0][0:2])
        minute=int(L[0][3:5])
        sec=int(L[0][6:8])
        ms=int(L[0][9:12])
        self.time=timeofday(hr,minute,sec,ms)
        self.customerid=L[1]
        self.activate_or_deactivate=L[2]
        self.bid_or_ask=L[3]
        self.orderid=L[4]
        if self.activate_or_deactivate=="activate":
            self.price=int(L[5])
            self.quantity=int(L[6])
        else:
            self.price=None
            self.quantity=None
        
    def __str__(self):
        time=self.time
        tstr=time.__str__()+","
        tstr+=self.customerid+","
        tstr+=self.activate_or_deactivate+","
        tstr+=self.bid_or_ask+","
        tstr+=self.orderid+","
        tstr+=str(self.price)+","
        tstr+=str(self.quantity)
        return(tstr)

#
# bid object 
#
class bid:
    def __init__(self,orderid,customerid,time,price,quantity,active):
        self.orderid=orderid
        self.customerid=customerid # index in customer array
        self.time=time
        self.price=price
        self.quantity=quantity
        # when bid is constructed it becomes active
        self.active=active
        # when creating a bid we give it a special_status flag
        #      depending on the customerid
        #
        self.special_status=customer_dictionary[customerid].special_status
#
# Problem 2. Write function to determine 
# bid priority between self bid and other bid.
# 
    def __lt__(self,other):
        if self.price == other.price:
            if self.special_status == other.special_status:
                return self.time < other.time
            else:
                return self.special_status > other.special_status
        else:
            return self.price > other.price
        ...
        
        
        
    # make string for bid object
    #
    def __str__(self):
        tstr="bid:"
        time=self.time
        tstr+=time.__str__()+" "
        tstr+=self.customerid+" "
        if self.active:
            tstr+=" active "
        else:
            tstr+=" inactive "
        tstr+=self.orderid+" "
        tstr+=str(self.price)+" "
        tstr+=str(self.quantity)+" "
        tstr+=str(self.special_status)+"\n"
        return(tstr)
#
# ask object
#
class ask:
    def __init__(self,orderid,customerid,time,price,quantity,active):
        self.orderid=orderid
        self.customerid=customerid # index in customer array
        self.time=time
        self.price=price
        self.quantity=quantity
        # initially ask is active
        self.active=active
        # assign special status
        self.special_status=customer_dictionary[customerid].special_status
#
# Problem 3. Write function to determine 
# priority between self ask and other ask
#
    def __lt__(self,other):
        if self.price == other.price:
            if self.special_status == other.special_status:
                return self.time < other.time
            else:
                return self.special_status > other.special_status
        else:
            return self.price < other.price
        ...
    #
    # make string for ask object
    #
    def __str__(self):
        tstr="ask:"
        time=self.time
        tstr+=time.__str__()+" "
        tstr+=str(self.customerid)+" "
        if self.active:
            tstr+=" active "
        else:
            tstr+=" inactive "
        tstr+=self.orderid+" "
        tstr+=str(self.price)+" "
        tstr+=str(self.quantity)+" "
        tstr+=str(self.special_status)+"\n"
        return(tstr)        
        
#
# Customer object
#
class customer:
    def __init__(self,customerid,special_status,nshares,cash,active_bidids,active_askids):
        self.customerid=customerid
        self.special_status=special_status
        self.nshares=nshares
        self.cash=cash
        self.active_bidids=active_bidids
        self.active_askids=active_askids
    def __str__(self):
        tstr=self.customerid+" "
        tstr+=str(self.nshares)+" "
        tstr+=str(self.cash)
        tstr+="\n"
        return(tstr)

    
#
# Problem 4. Write a function to 
# remove top priority unactive bids from the 
# bid queue until we get an active one at the top
# or the queue is empty. When we get an active one
# it should remain in the queue.
#
# Do the same with the ask queue - remove the 
# top priority asks until we get an active one at the 
# top. If there is an active ask at the top
# make sure it remains there.
#
def remove_top_priority_inactive_bids():
    global bid_queue
    while not bid_queue.empty() and not bid_queue.queue[0].active:
        bid_queue.get()
    ...
def remove_top_priority_inactive_asks():
    global ask_queue
    while not ask_queue.empty() and not ask_queue.queue[0].active:
        ask_queue.get()
    ...
    
#
# update the queues
#
def update_queue():
    global bid_queue,ask_queue,customer_dictionary,trades_processed, trade_desc
    
    while True:
        #
        # remove top priority bids and asks that
        # are inactive
        #
        remove_top_priority_inactive_bids()
        remove_top_priority_inactive_asks()     
        #
        # if either queue is empty we return
        #
        if bid_queue.empty():
            return
        if ask_queue.empty():
            return
        
        #
        # if either queue is empty we return
        #
        if bid_queue.empty():
            return
        if ask_queue.empty():
            return
#
# Problem 5. Assign bid to be the top priority bid and ask to the the top priority ask
#            Do this without removing then from the queue.
#
        bid = bid_queue.queue[0]
        ask = ask_queue.queue[0]
        
        if bid.price<ask.price: # no trade so we are done updating the queue
            return
        #
        # a trade should take place
        # so process it
        #
        trades_processed=trades_processed+1
        
        
        if DEBUG:
            forderbooks=open("orderbooks.txt","a")
            forderbooks.write("\nnew order book pre trade\n")
            print_orderbook(forderbooks)
            forderbooks.write("\n")
            forderbooks.close()
            
            
#
# Problem 6. Determine the price they trade at according to the rules
#            and call the price "current_price"
#
        price = 0
        if bid.special_status == ask.special_status:
            price = bid.price if bid.time < ask.time else ask.price
        else:
            price = bid.price if bid.special_status < ask.special_status else ask.price

        current_price = price

#
# Problem 7. Determine how many shares are traded for these two particular contracts.
#            Call this "quantity"
#
        quantity = min(bid.quantity, ask.quantity)
        
#
# Problem 8. Get the customer id's associated with these contracts
#            and assign to "bidder_customerid" and "asker_customerid"

        #
        # get customer ids
        #
        bidder_customerid = bid.customerid
        asker_customerid = ask.customerid
        
        
#
# Problem 9. Update the bidder and asker's cash and nshares in the customer_dictionary.
#            Note that the amount of cash exchanged is quantity*current_price
        cash_exchanged = quantity * current_price

        bcash_pre = customer_dictionary[bidder_customerid].cash
        bcash_post = bcash_pre - cash_exchanged
        customer_dictionary[bidder_customerid].cash = bcash_post
        
        bnshares_pre = customer_dictionary[bidder_customerid].nshares
        bnshares_post = bnshares_pre + quantity
        customer_dictionary[bidder_customerid].nshares = bnshares_post
        
        acash_pre = customer_dictionary[asker_customerid].cash
        acash_post = acash_pre + cash_exchanged
        customer_dictionary[asker_customerid].cash = acash_post
        
        anshares_pre = customer_dictionary[asker_customerid].nshares
        anshares_post = anshares_pre - quantity
        customer_dictionary[asker_customerid].nshares = anshares_post
        
        #
        # update message to print about trade
        #
        trade_desc="TRADE DESC: bidder "+bidder_customerid+ " asker "+asker_customerid+" price "+str(current_price) + " qty "+str(quantity)+"\n"
        trade_desc+="bidid = "+bid.orderid+" askid = "+ask.orderid+"\n"
        trade_desc+="cash exchanged = "+str(cash_exchanged)+"\n"
        trade_desc+="bidder cash pre-trade = "+str(bcash_pre)+" bidder cash post-trade "+str(bcash_post)+"\n"
        trade_desc+="bidder qty pre-trade = "+str(bnshares_pre)+" bidder qty post-trade "+str(bnshares_post)+"\n"
        trade_desc+="asker cash pre-trade = "+str(acash_pre)+" asker cash post-trade "+str(acash_post)+"\n"
        trade_desc+="asker qty pre-trade = "+str(anshares_pre)+" asker qty post-trade "+str(anshares_post)+"\n\n"
            
        ftrades=open("TRADES.csv","a")
        ftrades.write(trade_desc)
        ftrades.close()
            
        
        
        if DEBUG:
            forderbooks=open("orderbooks.txt","a")
            forderbooks.write("\nnew order book post trade\n")
            print_orderbook(forderbooks)
            forderbooks.write("\n")
            forderbooks.close()
        #
        # modify the bid and asks to reflect changes in quantities since a trade took
        # place - note that one of the contracts might remain active when done 
        # because the quantity traded is not the quantity in the bid or ask
        #       
        bid.quantity-=quantity
        ask.quantity-=quantity
        
#         print(customer_dictionary[asker_customerid].nshares,customer_dictionary[asker_customerid].cash,customer_dictionary[bidder_customerid].nshares,customer_dictionary[bidder_customerid].cash)
        
#
# Problem 10: if bid quantity becomes zero, you need to
#    - deactivate the bid
#    - remove it from the customer's active bid list
#    - remove the bid from the bid queue
#
        if bid.quantity==0:
            bid.active = False
            customer_dictionary[bidder_customerid].active_bidids.remove(bid.orderid)
            bid_queue.get()
        
            ...
#
# Problem 11: if ask quantity becomes zero, we need to
#    - deactivate it
#    - remove it from the customer's active ask list
#    - remove the ask from the bid queue
#
        if ask.quantity==0:
            ask.active = False
            customer_dictionary[asker_customerid].active_askids.remove(ask.orderid)
            ask_queue.get()
            ...
            
        if DEBUG:
            forderbooks=open("orderbooks.txt","a")
            forderbooks.write("\nnew order book post trade\n")
            print_orderbook(forderbooks)
            forderbooks.write("\n")
            forderbooks.close()
      
            
def print_orderbook(fout):
    global bid_queue, ask_queue, ctime
    
    bid_queue2=Q.PriorityQueue()
    ask_queue2=Q.PriorityQueue()
    
    blist=[]
    while not bid_queue.empty():
        bid=bid_queue.get()
        blist.append(bid)
        bid_queue2.put(bid)
    while not bid_queue2.empty():
        bid=bid_queue2.get()
        bid_queue.put(bid)
    
    
    alist=[]
    while not ask_queue.empty():
        ask=ask_queue.get()
        alist.append(ask)
        ask_queue2.put(ask)
    while not ask_queue2.empty():
        ask=ask_queue2.get()
        ask_queue.put(ask)
    
    #
    # compute top bid bottom ask
    #
    if len(blist)>0:
        topbid=max([bid.price for bid in blist])
    else:
        topbid=None
    
    if len(alist)>0:
        bottomask=min([ask.price for ask in alist])
    else:
        bottomask=None
    
    if topbid is None or bottomask is None:
        tradehappens=False
    else:
        if topbid>=bottomask:
            tradehappens=True
        else:
            tradehappens=False
    
    
    if tradehappens:
        fout.write("YES TRADE with topbid = "+str(topbid)+" bottomask = "+str(bottomask)+"\n")
    else:
        fout.write("NO TRADE should happen with topbid = "+str(topbid)+" bottomask = "+str(bottomask)+"\n")
    
    
    fout.write("\n")
    alist=list(reversed(alist))
    for a in alist:
        fout.write(a.__str__()+"\n")
    fout.write("------------------------------------------------------------------\n")
    for b in blist:
        fout.write(b.__str__()+"\n")
    fout.write("\n")

#
# given a new bid dataline instantiate a new bid and make it active
#
def activate_new_bid(req):
    global bid_queue, customer_dictionary
    
    # instantiate a new bid
    newbid=bid(req.orderid,req.customerid,req.time,req.price,req.quantity,True)
    
    #print(newbid)
    
    # add to the bid dictionary
    bid_dictionary[req.orderid]=newbid
    
    # put it in the queue
    bid_queue.put(newbid)
    
    # update the customer's active bid list
    customer_dictionary[req.customerid].active_bidids.append(req.orderid)
#
# given an new ask dataline instantiate a new ask and make it active
#      
def activate_new_ask(req):
    global ask_queue, customer_dictionary
    
    # instantiate a new ask
    newask=ask(req.orderid,req.customerid,req.time,req.price,req.quantity,True)
    # add to the ask dictionary
    ask_dictionary[req.orderid]=newask
    # put it in the queue
    ask_queue.put(newask)
    # update the customer's active ask list
    customer_dictionary[req.customerid].active_askids.append(req.orderid)
#
# given a cancel bid data line, deactivate the bid
#
def deactivate_bid(req):
    global bid_queue, customer_dictionary
    
    bidid=req.orderid
    customerid=req.customerid
    #
    # remove from the customer's bid list
    #
    #print("deactivating bid with id = "+bidid+" for customer "+customerid)
    oldlist=customer_dictionary[customerid].active_bidids
    #print(oldlist)
    newlist=[b for b in oldlist if b!=bidid]
    #print(newlist)
    #print("\n")
    customer_dictionary[customerid].active_bidids=newlist
    #
    # make bid inactive
    #
    bid_dictionary[bidid].active=False
    
#
# given a cancel ask data line, deactivate the ask
#
def deactivate_ask(req):
    global ask_queue, customer_dictionary
    askid=req.orderid
    customerid=req.customerid
    #
    # remove from the customer's bid list
    #
    oldlist=customer_dictionary[customerid].active_askids
    newlist=[a for a in oldlist if a!=askid]
    customer_dictionary[customerid].active_askids=newlist
    #
    # make bid inactive
    #
    ask_dictionary[askid].active=False
    #
    # remove the inactive asks from the queue
    #
    
#
# process a request
#
def process_request(req):
    if req.activate_or_deactivate=="activate":
        if req.bid_or_ask=="bid":
            activate_new_bid(req)
        else:
            activate_new_ask(req)
    else:
        if req.bid_or_ask=="bid":
            deactivate_bid(req)
        else:
            deactivate_ask(req)


#
# get the current bid queue as a string
#
def bid_queue_string():
    global bid_queue
    bid_queue_str=""
    qtemp=Q.PriorityQueue()
    for i in range(len(bid_queue.queue)):
        tbid = bid_queue.get()
        if tbid.active:
            bid_queue_str+=tbid.__str__()
        qtemp.put(tbid)
    bid_queue=qtemp
    return(bid_queue_str)
#
# get the current ask queue as a string
#
def ask_queue_string():
    global ask_queue
    ask_queue_str=""
    qtemp=Q.PriorityQueue()
    for i in range(len(ask_queue.queue)):
        task = ask_queue.get()
        if task.active:
            ask_queue_str+=task.__str__()
        qtemp.put(task)
    ask_queue=qtemp
    return(ask_queue_str)

#######################################################################
#######################################################################
#
# DO NOT MODIFY ANY OF THE FOLLOWING LINES
#
# When you run your code, the two files:
#
#   initial_customer_data.csv
#   requests.csv
#
# should be in the same folder as this notebook.
#
# load the customer data from initial customer data 
# make each customer into a customer instance and create a dictionary
# of all customers using their cid's as keys
#
#
customers=pd.read_csv("initial_customer_data.csv")
ncustomers=customers.shape[0]
for i in range(ncustomers):
    c=customer(customerid=customers.cid[i],
               special_status=customers.special_status[i],
               nshares=customers.nshares[i],
               cash=customers.cash[i],
               active_bidids=[],active_askids=[])
    cid=customers.cid[i]
    customer_dictionary[cid]=c
####################################################################
#
# process requests line by line
#
####################################################################
fin=open("requests.csv")
fin.readline() # skip the header row
req_counter=0
while True:   
    # read one line from file until we reach a blank line
    txt=fin.readline().replace("\n","")
    if txt=="":
        break
    # create a request object
    req=request(txt)
    req_counter+=1
    process_request(req)
    update_queue()
    if DEBUG:
        if req_counter==number_of_requests_to_process:
            break
    
    
print(req_counter)
############################################################################
#
# print out the customer data - you should not change any of these lines!!!
#
with open("final_customer_data.csv","w") as fout:
    fout.write("cid,nshares,cash\n")
    for cid in sorted(customer_dictionary.keys()):
        customer=customer_dictionary[cid]
        fout.write(cid+","+str(customer.nshares)+","+str(customer.cash)+"\n")
#################################################################################
#
# print out the final state of the bid queue
# this is for debugging purposes only - not handed in
#
with open("final_bid_queue_state.txt","w") as fout:
    bid_queue_str=bid_queue_string()
    fout.write(bid_queue_str)
    fout.write("\n")
#
# print out the final state of the ask queue
# this is for debugging purposes only - not handed in
#
with open("final_ask_queue_state.txt","w") as fout:
    ask_queue_str=ask_queue_string()
    fout.write(ask_queue_str)
    fout.write("\n")
print("done ")
print("trades processed = "+str(trades_processed))
